<a href="https://colab.research.google.com/github/asmaasultan202-prog/asmaasultan202-prog/blob/main/loan_approval_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd

# قراءة ملف البيانات بنفس الاسم اللي ظهر في القايمة
df = pd.read_csv('/loan_data.csv')

# عرض أول 5 صفوف عشان نتأكد إن البيانات اتقرأت صح
df.head()

,Text,Income,Credit_Score,Loan_Amount,DTI_Ratio,Employment_Status,Approval
0,I need a loan to pay for an international vaca...,26556,581,8314,79.26,employed,Rejected
1,I want to make home improvements like installi...,197392,389,111604,22.14,employed,Rejected
2,"I need a loan for home renovation, including a...",44561,523,34118,45.44,employed,Rejected
3,I need funds to buy new furniture and applianc...,190363,729,118757,10.22,unemployed,Rejected
4,I need a loan to start a small business.,61853,732,19210,44.13,employed,Approved


In [7]:
# الكود الأول: لمعرفة عدد الصفوف وأنواع البيانات
print(df.info())

print("-" * 50)

# الكود الثاني: لمعرفة لو فيه خانات فاضية (Missing Values)
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24000 entries, 0 to 23999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Text               24000 non-null  object 
 1   Income             24000 non-null  int64  
 2   Credit_Score       24000 non-null  int64  
 3   Loan_Amount        24000 non-null  int64  
 4   DTI_Ratio          24000 non-null  float64
 5   Employment_Status  24000 non-null  object 
 6   Approval           24000 non-null  object 
dtypes: float64(1), int64(3), object(3)
memory usage: 1.3+ MB
None
--------------------------------------------------
Text                 0
Income               0
Credit_Score         0
Loan_Amount          0
DTI_Ratio            0
Employment_Status    0
Approval             0
dtype: int64


In [8]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. حذف عمود الـ Text
df.drop('Text', axis=1, inplace=True)

# 2. تحويل البيانات الفئوية لأرقام
le_emp = LabelEncoder()
df['Employment_Status'] = le_emp.fit_transform(df['Employment_Status'])

le_app = LabelEncoder()
# Approved هياخد 0 و Rejected هياخد 1 (أو العكس حسب الترتيب الأبجدي)
df['Approval'] = le_app.fit_transform(df['Approval'])

# 3. فصل الميزات (X) عن الهدف اللي عايزين نتوقعه (y)
X = df.drop('Approval', axis=1)
y = df['Approval']

# 4. عمل Scaling للميزات عشان اللوجيستيك ريجريشن
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# عرض أول 5 صفوف بعد التجهيز
print(X_scaled.head())
print("-" * 50)
print("Target Variable (y) head:")
print(y.head())

     Income  Credit_Score  Loan_Amount  DTI_Ratio  Employment_Status
0 -1.620410      0.033159    -1.039701   1.378043          -1.000584
1  1.682134     -1.172687     1.939886  -0.389185          -1.000584
2 -1.272344     -0.331107    -0.295338   0.331690          -1.000584
3  1.546252      0.962665     2.146227  -0.757977           0.999417
4 -0.938060      0.981506    -0.725386   0.291161          -1.000584
--------------------------------------------------
Target Variable (y) head:
0    1
1    1
2    1
3    1
4    0
Name: Approval, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# 1. تقسيم البيانات لتدريب (80%) واختبار (20%)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 2. تعريف الـ 3 موديلز
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

# 3. تدريب الموديلز وحساب الدقة
results = []

for name, model in models.items():
    # التدريب على بيانات الـ Train
    model.fit(X_train, y_train)

    # التوقع لبيانات الـ Train والـ Test
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # حساب نسبة الدقة (Accuracy)
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    # حفظ النتائج
    results.append({
        'Model': name,
        'Train Accuracy': train_acc,
        'Test Accuracy': test_acc
    })

# 4. تحويل النتائج لـ DataFrame وعرضها
final_comparison = pd.DataFrame(results)
print(final_comparison)

                 Model  Train Accuracy  Test Accuracy
0  Logistic Regression        0.930208       0.928333
1        Decision Tree        1.000000       0.996042
2        Random Forest        1.000000       0.998125


In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss
from sklearn.model_selection import GridSearchCV
import pandas as pd

# القاموس اللي هنجمع فيه كل النتائج
results_dict = []

# دالة مساعدة عشان نحسب كل المقاييس ونضيفها في القاموس وتوفر علينا تكرار الكود
def evaluate_and_store(model_name, state, model, X_train, y_train, X_test, y_test):
    # التوقع
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test) # ضروري لحساب الـ Log Loss

    # حساب المقاييس
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    loss = log_loss(y_test, y_prob)

    # تخزين النتائج
    results_dict.append({
        'Model': model_name,
        'State': state, # Before Tuning أو After Tuning
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'Log Loss': round(loss, 4)
    })

In [12]:
from sklearn.linear_model import LogisticRegression

print("Training Logistic Regression...")
# 1. Before Tuning
lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_base.fit(X_train, y_train)
evaluate_and_store('Logistic Regression', 'Before Tuning', lr_base, X_train, y_train, X_test, y_test)

# 2. Grid Search (Tuning)
lr_params = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear']
}
lr_grid = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42), lr_params, cv=5, scoring='accuracy')
lr_grid.fit(X_train, y_train)

# 3. After Tuning
lr_best = lr_grid.best_estimator_
evaluate_and_store('Logistic Regression', 'After Tuning', lr_best, X_train, y_train, X_test, y_test)

Training Logistic Regression...


In [13]:
from sklearn.tree import DecisionTreeClassifier

print("Training Decision Tree...")
# 1. Before Tuning
dt_base = DecisionTreeClassifier(random_state=42)
dt_base.fit(X_train, y_train)
evaluate_and_store('Decision Tree', 'Before Tuning', dt_base, X_train, y_train, X_test, y_test)

# 2. Grid Search (Tuning)
dt_params = {
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}
dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_params, cv=5, scoring='accuracy')
dt_grid.fit(X_train, y_train)

# 3. After Tuning
dt_best = dt_grid.best_estimator_
evaluate_and_store('Decision Tree', 'After Tuning', dt_best, X_train, y_train, X_test, y_test)

Training Decision Tree...


In [15]:
from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest...")
# 1. Before Tuning
rf_base = RandomForestClassifier(random_state=42)
rf_base.fit(X_train, y_train)
evaluate_and_store('Random Forest', 'Before Tuning', rf_base, X_train, y_train, X_test, y_test)

# 2. Grid Search (Tuning)
rf_params = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}
# قللنا الخيارات شوية عشان وقت التدريب مياخدش كتير على كولاب
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5, scoring='accuracy')
rf_grid.fit(X_train, y_train)

# 3. After Tuning
rf_best = rf_grid.best_estimator_
evaluate_and_store('Random Forest', 'After Tuning', rf_best, X_train, y_train, X_test, y_test)

Training Random Forest...


In [16]:
# تحويل القاموس لجدول وعرض النتيجة
final_df = pd.DataFrame(results_dict)

print("=== المقارنة النهائية (Before vs After Grid Search) ===")
# الكود ده بيعرض الجدول بشكل منظم جداً في كولاب
display(final_df)

=== المقارنة النهائية (Before vs After Grid Search) ===


,Model,State,Accuracy,Precision,Recall,F1-Score,Log Loss
0,Logistic Regression,Before Tuning,0.9283,0.9584,0.9565,0.9575,0.1482
1,Logistic Regression,After Tuning,0.9281,0.9584,0.9563,0.9573,0.1482
2,Logistic Regression,Before Tuning,0.9283,0.9584,0.9565,0.9575,0.1482
3,Logistic Regression,After Tuning,0.9281,0.9584,0.9563,0.9573,0.1482
4,Decision Tree,Before Tuning,0.9960,0.9988,0.9965,0.9976,0.1427
5,Decision Tree,After Tuning,0.9950,0.9980,0.9960,0.9970,0.1509
6,Random Forest,Before Tuning,0.9981,0.9993,0.9985,0.9989,0.0147
7,Random Forest,Before Tuning,0.9981,0.9993,0.9985,0.9989,0.0147
8,Random Forest,After Tuning,0.9977,0.9988,0.9985,0.9986,0.0145


In [17]:
final_df.to_csv('models_comparison.csv', index=False)